# Evaluate models

This notebook evaluates a model's predictions against SAGE and reproduces the paper's numbers: per-species metrics, the sampling-aware stratified report, and the delta heatmap.

Every checkpoint from the paper is available, so you can evaluate and compare them directly. We'll walk through the workflow on those provided models, but the same commands work on your own `test_predictions.h5`, say from [02 · Train a Model](02_train_a_model.ipynb).

In [1]:
import os
import glob
import yaml
import h5py
import numpy as np
import pandas as pd
import rootutils
ROOT = rootutils.setup_root(os.path.abspath(""), indicator=".project-root", pythonpath=True, cwd=True)
log_dir = yaml.safe_load(open("configs/local_default.yaml"))["log_dir"]
print("✅ ready")

✅ ready


## The models we provide

Every reported model is trained with 5 seeds. For each run you get the checkpoint, `test_predictions.h5`, and `metrics_by_species.h5`, in a folder named `<model>_s<seed>`. They're all listed in `configs/paper_runs.yaml`. The DeepSDMs form a bias-correction ablation ladder, where each row adds one practice on top of the one before:

In [2]:
models = yaml.safe_load(open("configs/paper_runs.yaml"))["models"]
pd.DataFrame([
    {"paper model": m, "what it adds": d.get("note", ""),
     "seeds": len(d["runs"]), "example run id": d["runs"][0]}
    for m, d in models.items()
])

,paper model,what it adds,seeds,example run id
0,MaxEnt,"MaxEnt (maxnet), target-group background",5,maxent_s0
1,Random Forest,down-sampled Random Forest (Valavi et al. 2021...,5,randomforest_s0
2,Baseline DeepSDM,"plain DeepSDM, no bias correction",5,baseline_deepsdm_s0
3,+Aggregation,+ 1 km equal-area aggregation of occurrences,5,aggregation_s0
4,+Subsampling,+ occurrence subsampling (N = 1000 per species),5,subsampling_s0
5,+Architecture,+ ResNet architecture,5,architecture_s0
6,Optimized DeepSDM,+ loss reweighting (the best model; includes a...,5,optimized_deepsdm_s0


## The prediction format

A predictions file evaluates every location for every species. We recommend HDF5:

| dataset | shape | meaning |
|---|---|---|
| `probabilities` | `(N_locations, N_species)` | predicted scores in `[0, 1]`, **required** |
| `locations` | `(N_locations, 2)` | `[longitude, latitude]`, optional |
| `targets` | `(N_locations, N_species)` | ground truth `{0, 1}`, `-1` = ignore, optional |

Species follow the canonical order in `targets/species_names.csv`. This is exactly what a trained run writes. Here's a real provided run (42,268 evaluation plots by 5,771 species):

In [3]:
found = glob.glob(os.path.join(log_dir, "**", "test_predictions.h5"), recursive=True)
peek = found[0] if found else "examples/example_predictions.h5"
print("peeking:", os.path.relpath(peek))
with h5py.File(peek) as f:
    for k in f:
        print(f"  {k:14s} {str(f[k].shape):18s} {f[k].dtype}")

peeking: ../logs/sage_benchmark/ib5d7d81/test_predictions.h5
  locations      (42268, 2)         float32
  probabilities  (42268, 5771)      float32
  targets        (42268, 5771)      float32


## Bring your own predictions

Any model works. Run it on the evaluation locations and write a file in the format shown above.

In [4]:
from src.evaluation.prediction_io import load_predictions
os.makedirs("examples/output", exist_ok=True)
N, S = 200, 50                                    # locations, species (canonical order)
probs = np.random.rand(N, S).astype("float32")    # your model's scores in [0, 1]
with h5py.File("examples/output/my_predictions.h5", "w") as f:
    f["probabilities"] = probs
    f["locations"] = np.random.uniform(-60, 60, (N, 2)).astype("float32")   # optional
    f["targets"]   = (np.random.rand(N, S) > 0.85).astype("float32")        # optional {0,1}
print("validated:", {k: (None if v is None else v.shape) for k, v in load_predictions("examples/output/my_predictions.h5").items()})

validated: {'probabilities': (200, 50), 'locations': (200, 2), 'targets': (200, 50)}


## Evaluate and compare two runs

One command computes per-species metrics, a macro summary, and, with `--reference`, the delta against another model, along with the stratified breakdown and delta heatmap. Pick any runs from the table above.

```bash
# Optimized DeepSDM (seed 2) against the Baseline DeepSDM (seed 2)
python scripts/evaluate_predictions.py \
    --predictions logs/optimized_deepsdm_s2/test_predictions.h5 \
    --reference   logs/baseline_deepsdm_s2/test_predictions.h5 \
    --data-dir    <data_dir> \
    --metrics auroc auprg \
    --output-dir  results/
```

`--targets` is optional: without it the ground truth is read from `<data_dir>/targets/eval_test_targets.h5`, which is the same file shipped alongside the released runs.

> These prediction files are ~1.9 GB each (42 k plots × 5,771 species), so the full eval needs the downloaded runs and a good amount of RAM. It's a command-line step rather than an inline cell. The output below is the real result of that command.

It prints:

```
metric        macro(%)   reference   delta
auroc          85.834     83.394  +2.440
auprg          59.959     56.985  +2.973
auprg(median)   84.912     83.308  +1.604

Stratified delta vs reference (M-ref) (%) — x=sampling_effort [sparse<->dense], y=relative_prevalence [infrequent<->frequent]
  stratum                            n       auroc     auprg
  -- quadrants (median split) --
  sparse-infrequent               1421        4.04      9.57
  sparse-frequent                 1464        3.35      6.62
  dense-infrequent                1464        1.32     -0.07
  dense-frequent                  1422        1.06     -4.24
  -- extreme L-strips (p10 x median split, matches heatmap) --
  sparse(p10) x infrequent-half    235        6.63     18.32
  sparse(p10) x frequent-half      325        4.72      0.27
  infrequent(p10) x sparse-half    207        3.54     21.99
  infrequent(p10) x dense-half     357        1.80     10.73
```

Two things to read carefully.

**AUPRG is reported twice.** AUPRG is unbounded below, so a handful of species drag the macro *mean* far down; the paper reports the *median* over species, which is the `auprg(median)` line (84.9 vs 83.3 here). Use the mean only if you want the outlier tail to count.

**Read the stratified block, not just the macro line.** Bias correction lifts AUROC in every group, and the gain is largest where you'd expect: sparse, infrequently recorded species (+4.04), smallest for dense, frequent ones (+1.06). The AUPRG deltas are not uniformly positive — the correction buys a large gain in the sparse half (+9.6 / +6.6) and costs a little in the dense-frequent corner (-4.2). That trade is exactly what a single average would hide.

## The delta heatmap

The stratified deltas rendered over the property space. Green means this run (Optimized) wins, purple means the reference (Baseline) wins. It's all green here, deepening toward the sparse-infrequent corner:

<img src="assets/example_delta_heatmap.png" width="440" alt="Delta AUROC heatmap: Optimized vs Baseline DeepSDM, largest gains for sparse-infrequent species">

## The per-species table

`species_metrics.csv`, written next to the heatmap, has one row per species with both models and the delta, in case you want to run your own analysis:

In [ ]:
sm = pd.read_csv("tutorials/assets/example_species_metrics.csv")   
print(sm.shape[0], "species ·", ", ".join(sm.columns[:6]), "...")
sm.head()

5771 species · species_index, species_name, auroc, auprg, auroc_reference, auroc_delta ...


,species_index,species_name,auroc,auprg,auroc_reference,auroc_delta,auprg_reference,auprg_delta
0,0,Abies alba,0.783674,0.749127,0.766751,0.016923,0.709099,0.040028
1,1,Abies amabilis,0.970563,0.988531,0.978541,-0.007978,0.995423,-0.006892
2,2,Abies balsamea,0.753480,0.464662,0.737729,0.015751,0.476317,-0.011655
3,3,Abies concolor,0.928084,0.979749,0.896996,0.031089,0.963170,0.016579
4,4,Abies fraseri,0.983897,0.998953,0.979914,0.003983,0.995786,0.003168


## Where to next

- Compare more pairs from `configs/paper_runs.yaml`, for example *Optimized DeepSDM* against *Random Forest* (the cross-paradigm headline), or step through the ablation ladder.
- [04 · Generate Maps](04_generate_maps.ipynb) turns a provided model into continental maps.